In [1]:
from pyspark.sql import SparkSession

# 1. Login into Spark
spark = SparkSession.builder \
    .appName("NYC_Taxi_Stage1") \
    .getOrCreate()

# Extract the SparkContext (obligatory for the rubrik to read at low level)
sc = spark.sparkContext

# Mute the "Warnings" from Spark to have a clean output
sc.setLogLevel("ERROR")

print("Reading raw data...")

# 2. Read the text file 
file_path = "../data/raw/yellow_tripdata_2016-03.csv" 
raw_rdd = sc.textFile(file_path)

# Count the original lines
total_start = raw_rdd.count()
print(f"-> Total lines in the raw file: {total_start}")

# 3. TRANSFORMATION 1: mapPartitions (deleting the header)
def delete_header(index, partition):
    if index == 0: # If we are in the first block of the file...
        next(partition) # ...we skip the fist row
    return partition

no_header_rdd = raw_rdd.mapPartitionsWithIndex(delete_header)

# 4. TRANSFORMATION 2: map (Split by commas and convert it into a tuple)
# Since we are going to add tuple(), we acomplish with the requirement of "Output: an RDD whose elemnts are tuples"
split_rdd = no_header_rdd.map(lambda line: tuple(line.split(",")))

# 5. TRANSFORMATION 3: filter (Delete corrupt/malformed registers)
# The Yellow Taxis of 2016´s dataset has exactly 19 columns.
# If a row has more or less columns due to a writing error, we delete it.
cleaned_rdd = split_rdd.filter(lambda columns: len(columns) == 19)

# 6. The final report that the professor requests (take 5 and count)
cleaned_total = cleaned_rdd.count()
malformed_rows = total_start - 1 - cleaned_total # The -1 is for the header

print("\n=== INGESTION REPORT (STAGE 1) ===")
print(f"Discarted malformed rows: {malformed_rows}")
print(f"Total number of clean registers: {cleaned_total}")
print("\n--- SAMPLE OF 5 REGISTERS (.take(5)) ---")

for register in cleaned_rdd.take(5):
    print(register)

26/05/27 02:11:27 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:11:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:11:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Reading raw data...


-> Total lines in the raw file: 12210953



[Stage 1:=======================>                                (24 + 16) / 57]




[Stage 1:=============================================>          (46 + 11) / 57]




=== INGESTION REPORT (STAGE 1) ===
Discarted malformed rows: 0
Total number of clean registers: 12210952

--- SAMPLE OF 5 REGISTERS (.take(5)) ---


('1', '2016-03-01 00:00:00', '2016-03-01 00:07:55', '1', '2.50', '-73.97674560546875', '40.765151977539062', '1', 'N', '-74.004264831542969', '40.746128082275391', '1', '9', '0.5', '0.5', '2.05', '0', '0.3', '12.35')
('1', '2016-03-01 00:00:00', '2016-03-01 00:11:06', '1', '2.90', '-73.983482360839844', '40.767925262451172', '1', 'N', '-74.005943298339844', '40.733165740966797', '1', '11', '0.5', '0.5', '3.05', '0', '0.3', '15.35')
('2', '2016-03-01 00:00:00', '2016-03-01 00:31:06', '2', '19.98', '-73.782020568847656', '40.644809722900391', '1', 'N', '-73.974540710449219', '40.675769805908203', '1', '54.5', '0.5', '0.5', '8', '0', '0.3', '63.8')
('2', '2016-03-01 00:00:00', '2016-03-01 00:00:00', '3', '10.78', '-73.863418579101562', '40.769813537597656', '1', 'N', '-73.969650268554688', '40.757766723632812', '1', '31.5', '0', '0.5', '3.78', '5.54', '0.3', '41.62')
('2', '2016-03-01 00:00:00', '2016-03-01 00:00:00', '5', '30.43', '-73.97174072265625', '40.792182922363281', '3', 'N', '-7